# Phase 16: Encoding & Scaling Pipeline

**Goal:** Our data is mathematically clean, but it is not yet ready for an AI. 
AI models only understand numbers, but we have text columns like `protocol_type` (TCP/UDP). Furthermore, our numeric columns have wildly different scales (e.g., `duration` might be 5 seconds, but `bytes` might be 1,000,000). 

In this phase, we will convert text into numbers (**Encoding**) and squish massive numbers into a balanced range (**Scaling**).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, RobustScaler

### Step 1: The Golden Rule - Split Before Scaling
Before we do *any* scaling, we must split our data into a **Training Set** and a **Testing Set**. 

**Why? (Preventing Data Leakage)**: If you scale the entire dataset at once, the mathematical formula will "peek" at the testing data to calculate the average. This means your AI will cheat on the final exam because it has already seen hints from the test set! We only fit our scalers on the Training Data.

In [ ]:
# 1. Create a fake, clean mini-dataset to demonstrate
df = pd.DataFrame({
    "protocol_type": ["tcp", "udp", "tcp", "icmp", "tcp", "udp", "icmp", "tcp"],
    "bytes_sent": [500, 20, 999999, 50, 1000, 10, 15, 200],
    "label": [0, 0, 1, 0, 1, 0, 0, 0]
})

print("=== ORIGINAL CLEAN DATA ===")
display(df)

# 2. Split the data! (80% Train, 20% Test)
X = df.drop(columns=['label'])
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("\nSUCCESS! Data successfully split to prevent Data Leakage.")


### Step 2: Categorical Encoding (Text to Numbers)
AI models like Neural Networks cannot read the word `tcp`. We need to use a **Label Encoder** to convert `tcp`, `udp`, and `icmp` into numbers like `0`, `1`, and `2`.

In [ ]:
encoder = LabelEncoder()

# We FIT the encoder ONLY on the training data
X_train['protocol_type'] = encoder.fit_transform(X_train['protocol_type'])

# We TRANSFORM the test data (so it uses the exact same numbering system)
X_test['protocol_type'] = encoder.transform(X_test['protocol_type'])

print("=== AFTER ENCODING ===")
print("The 'protocol_type' column is now numbers!")
display(X_train)

### Step 3: Robust Scaling (Protecting the Outliers)
In Phase 15, we learned that we should **NOT** delete outliers, because in cybersecurity, outliers are often the actual cyber attacks (like a DDoS attack generating 999,999 bytes).

Standard scalers (like `StandardScaler`) get completely ruined by these massive numbers. Instead, we use `RobustScaler`. It scales the normal traffic perfectly, while allowing the hacker outliers to remain massive so the AI can easily detect them!

In [ ]:
scaler = RobustScaler()

# Again, we only FIT on the training data
numeric_columns = ['bytes_sent']
X_train[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])

# Transform the test data
X_test[numeric_columns] = scaler.transform(X_test[numeric_columns])

print("=== AFTER ROBUST SCALING ===")
print("Notice how the massive DDoS outlier (999,999) is successfully preserved as a huge number (1332.33),")
print("while the normal traffic is nicely scaled down near 0!")
display(X_train)